# Comparativa final

Todos los modelos resuelven el mismo problema: **clasificación supervisada multiclase**. La variable `id_genero` toma 10 valores y cada canción pertenece a una sola clase, con el mismo número de canciones en cada género. El reparto de train y test es el común del grupo: 800 y 200 canciones, estratificado por género y separado por `id_cancion`, para que los fragmentos de un mismo tema no crucen de un conjunto al otro. La clase asignada es siempre la de mayor probabilidad o la de mayor voto. No se exige una probabilidad mínima.

Las métricas son el accuracy, el F1 macro y el **tiempo de entrenamiento** del modelo que luego se evalúa. Con las clases equilibradas, el accuracy resume el acierto global. El F1 macro da el mismo peso a los 10 géneros.


## Modelos de machine learning

Los modelos clásicos se han elegido porque encajan con una clasificación en 10 clases.

Random Forest, XGBoost y LightGBM construyen fronteras no lineales sobre variables numéricas. El análisis exploratorio había mostrado solapamiento entre géneros y redundancia entre características espectrales: los árboles toleran ese solapamiento mejor que un modelo lineal. El Random Forest y el LightGBM clasifican cada fragmento de 3 segundos de la tabla macro y, con el voto mayoritario de esos fragmentos, obtienen también el género de la canción. El ajuste del Random Forest que se conserva tarda **12 s**. El del LightGBM, **27 s**. El XGBoost trabaja directamente sobre las características de 30 segundos, devuelve una etiqueta por canción y su entrenamiento final tarda **9 s**.

El KNN asigna el género mayoritario entre los vecinos más cercanos en el espacio de características de 30 segundos. El Naive Bayes gaussiano estima, para cada género, cómo se distribuyen esas características y se queda con la clase más probable. Los dos comparan el mismo problema y la misma entrada. La celda de cada ajuste marca **0 s**: el contador no muestra fracciones, así que el entrenamiento de los dos termina en menos de un segundo.


## Redes recurrentes y los fragmentos de 3 segundos

Las redes recurrentes se han elegido porque sirven para series temporales, y aquí el género no está en un único instante: se manifiesta a lo largo de la pista. Cada canción está partida en ventanas sucesivas de 3 segundos. Esas ventanas, leídas en el orden del archivo, son una serie temporal corta del mismo tema. Por eso la GRU y la RNN simple usan los datos de 3 segundos y no el resumen estático de 30 segundos: si ese resumen se repitiera en cada paso, la red dejaría de modelar el tiempo. Las dos devuelven **un género por canción**.

La GRU tarda **12 s** en ajustarse. La RNN simple, también **12 s**.


## La red neuronal artificial

La ANN no lee una serie. Cada canción entra como un solo vector de características de 30 segundos, la misma representación que usa el XGBoost, y unas capas densas le asignan una de las 10 clases. No hay estado que pase de un instante al siguiente. El ajuste de las 50 épocas dura unos **16 s**.

En las 200 canciones de prueba alcanza un **70,5 %** de accuracy y un F1 macro de **0,70**. En entrenamiento la precisión sube al 93,3 % (pérdida 0,2682 frente a 0,9208 en prueba), así que hay sobreajuste, aunque el resultado de test se mantiene por encima del 70 %. Sobre ese mismo vector de 30 segundos, el XGBoost llega al 78 % de accuracy y a un F1 macro de 0,77 en **9 s**.


## Diferencias entre la RNN simple, la LSTM y la GRU

Las tres son redes recurrentes, y las tres se quedan con la clase de probabilidad más alta. Cambia la celda, qué se considera un paso de la serie y cuánto tarda el ajuste.

**RNN simple.** En cada paso combina el fragmento actual con el estado del paso anterior. No tiene compuertas: al avanzar en la serie, el contexto de los primeros fragmentos pesa menos. En este proyecto son dos capas apiladas. Leen, en orden, los fragmentos de 3 segundos de `df_macro` (sin las columnas de 30 segundos) y predicen un género por canción. En las 200 canciones de prueba: accuracy **68,5 %** y F1 macro **0,68**. El entrenamiento dura **12 s**.

**LSTM.** Añade una memoria (el estado de celda) y tres compuertas, que deciden qué se olvida, qué se escribe y qué se lee. Así puede conservar información durante más pasos que una RNN simple. En este proyecto la LSTM no lee la canción como serie de fragmentos: cada ventana de 3 segundos es una muestra, y los pasos de la secuencia son las 59 características de audio de esa ventana. La etiqueta es la del fragmento. Sobre los 2000 fragmentos de prueba: accuracy **53,8 %** y F1 macro **0,54**. Es el ajuste más largo del grupo: **15 min 17 s**.

**GRU.** También guarda memoria, con dos compuertas (actualización y reinicio) y menos parámetros que la LSTM. Aquí es una sola capa. Lee la secuencia de fragmentos de 3 segundos y predice un género por canción, como la RNN simple. En las 200 canciones de prueba: accuracy **72 %** y F1 macro **0,72**. El entrenamiento dura **12 s**.

Entre las dos redes que sí recorren la canción en el tiempo, la GRU queda por encima de la RNN simple y tarda lo mismo. La LSTM responde a otra unidad (el fragmento) y a otra definición de paso (la característica, no el instante de la pista): su 53,8 % no se pone en la misma fila que el 72 % de la GRU, y además tarda del orden de un minuto por cada segundo de la GRU.


## Tiempos de entrenamiento

El tiempo que entra en la tabla es el del modelo que luego se evalúa.

| Modelo | Tiempo de entrenamiento |
| --- | --- |
| KNN | < 1 s |
| Naive Bayes | < 1 s |
| XGBoost | 9 s |
| Random Forest | 12 s |
| GRU | 12 s |
| RNN simple | 12 s |
| ANN | 16 s |
| LightGBM | 27 s |
| LSTM | 15 min 17 s |

En Random Forest, XGBoost, LightGBM y GRU es la duración de la celda de `fit`. En la RNN simple y en la LSTM es la suma de los tiempos por época que imprime Keras. En la ANN casi todas las épocas salen como 0 s, así que los 16 s se estiman con los milisegundos por paso de ese mismo registro. En el KNN y en el Naive Bayes la celda marca 0 s: ese contador redondea a cero lo que dura menos de un segundo, y el tiempo de la tabla queda en menos de 1 s.

La búsqueda de hiperparámetros queda fuera de esa columna. En Random Forest duró unos 14 minutos y el modelo ajustado se descartó porque en test bajaba a 0,778; se conserva el base, de 12 s. En XGBoost el GridSearch duró unos 70 minutos y el `fit` final, ya con los parámetros elegidos, 9 s. En LightGBM la búsqueda se cortó a los 32 minutos sin terminar.

El KNN y el Naive Bayes quedan por debajo de 1 s. El resto de los ajustes cortos se agrupa entre 9 y 27 segundos. La LSTM se separa: 15 minutos y 17 segundos para un accuracy de 0,538 por fragmento.


## Tabla de resultados

Accuracy, F1 macro y tiempo de entrenamiento en el conjunto de prueba. La primera tabla es el género de la canción. La segunda es el acierto sobre cada fragmento de 3 segundos.

### Por canción (200 canciones de test)

| Modelo | Familia | Entrada | Accuracy | F1 macro | Tiempo de entrenamiento |
| --- | --- | --- | --- | --- | --- |
| Random Forest, voto mayoritario | ML | Fragmentos de 3 s, agregados por canción | 0,81 | 0,81 | 12 s |
| XGBoost | ML | Características de 30 s | 0,78 | 0,77 | 9 s |
| LightGBM, voto mayoritario | ML | Fragmentos de 3 s, agregados por canción | 0,745 | 0,741 | 27 s |
| GRU | DL | Secuencia temporal de fragmentos de 3 s | 0,72 | 0,72 | 12 s |
| KNN | ML | Características de 30 s | 0,705 | 0,70 | < 1 s |
| ANN | DL | Características de 30 s | 0,705 | 0,70 | 16 s |
| RNN simple | DL | Secuencia temporal de fragmentos de 3 s | 0,685 | 0,68 | 12 s |
| Naive Bayes | ML | Características de 30 s | 0,525 | 0,50 | < 1 s |

### Por fragmento de 3 segundos (2000 fragmentos de test)

| Modelo | Familia | Qué es un paso de la serie | Accuracy | F1 macro | Tiempo de entrenamiento |
| --- | --- | --- | --- | --- | --- |
| Random Forest | ML | No es una serie: un vector por fragmento | 0,804 | 0,80 | 12 s |
| LightGBM | ML | No es una serie: un vector por fragmento | 0,742 | 0,737 | 27 s |
| LSTM | DL | Cada característica de audio del fragmento | 0,538 | 0,54 | 15 min 17 s |

El Random Forest por fragmento es el modelo base. En LightGBM, la media de probabilidades por canción (accuracy 0,74 y F1 macro 0,736) no superó al voto mayoritario, y es el voto el que entra en la tabla.


## Qué algoritmo es mejor

El algoritmo elegido es el **Random Forest con voto mayoritario**. Sobre las 200 canciones de prueba alcanza un accuracy de 0,81 y un F1 macro de 0,81, por encima del resto de modelos que responden al género del tema, y su ajuste dura **12 s**.

Esa elección encaja con el problema. Diez clases equilibradas y características numéricas con solapamiento entre géneros favorecen a un conjunto de árboles. El voto de los fragmentos de 3 segundos convierte esas predicciones en una etiqueta por canción, que es la unidad del enunciado.

Dentro de las redes, la comparación útil es por el tipo de entrada. Sobre el vector de 30 segundos, el XGBoost (0,78 y 0,77, **9 s**) supera al KNN y a la ANN, que quedan empatados en 0,705 y 0,70. El KNN termina en menos de 1 s y la ANN tarda unos **16 s**. El Naive Bayes, con la misma entrada, también termina en menos de 1 s y se queda en 0,525 y 0,50. Sobre la serie temporal de fragmentos de 3 segundos, la GRU (0,72 y 0,72, **12 s**) supera a la RNN simple (0,685 y 0,68, **12 s**). La LSTM, por fragmento, queda en 0,538 y 0,54 después de **15 min 17 s**.
